<a href="https://colab.research.google.com/github/11310238-glitch/test20261104/blob/main/%E5%BE%8C%E7%AB%AF%E6%B8%AC%E8%A9%A6_1104.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# 1. 安裝 pyngrok 與 flask
!pip install pyngrok flask flask-cors

from flask import Flask, jsonify
from flask_cors import CORS
from pyngrok import ngrok

app = Flask(__name__)
CORS(app)  # 允許跨來源請求

@app.route('/')
def home():
    return jsonify({"status": "success", "message": "太棒了！後端 API 成功連線！"})

# 關閉之前殘留的 ngrok 通道避免衝突
ngrok.kill()

# 建立 ngrok 公開通道 (Port 5001)
public_url = ngrok.connect(5001).public_url
print("==================================================")
print(f"👉 複製這串最新的公開網址：{public_url}")
print("==================================================")

# 啟動 Flask 伺服器 (改用 Port 5001 避免衝突)
if __name__ == '__main__':
    app.run(port=5001)

👉 複製這串最新的公開網址：https://overhead-hatbox-yiddish.ngrok-free.dev
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5001
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [04/Nov/2026 11:24:15] "OPTIONS / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [04/Nov/2026 11:24:16] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [04/Nov/2026 11:46:10] "OPTIONS / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [04/Nov/2026 11:46:10] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [04/Nov/2026 11:46:11] "OPTIONS / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [04/Nov/2026 11:46:11] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [04/Nov/2026 11:46:13] "OPTIONS / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [04/Nov/2026 11:46:13] "GET / HTTP/1.1" 200 -


In [15]:
# 1. 安裝必要套件
!pip install pyngrok flask flask-cors

from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import sqlite3

app = Flask(__name__)
CORS(app)

# 初始化 SQLite 資料庫與表格
def init_db():
    conn = sqlite3.connect('survey_data.db')
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS results (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            zodiac TEXT,
            disc TEXT
        )
    ''')
    conn.commit()
    conn.close()

init_db()

# 接收前端 POST 傳來的資料
@app.route('/submit', methods=['POST'])
def submit():
    data = request.get_json()
    zodiac = data.get('zodiac')
    disc = data.get('disc')

    # 寫入 SQL 資料庫
    conn = sqlite3.connect('survey_data.db')
    cursor = conn.cursor()
    cursor.execute('INSERT INTO results (zodiac, disc) VALUES (?, ?)', (zodiac, disc))
    conn.commit()
    conn.close()

    return jsonify({"status": "success", "message": f"已成功記錄您的資料：{zodiac} + {disc}"})

# 額外加開一個路由讓你可以隨時查看統計結果
@app.route('/stats', methods=['GET'])
def stats():
    conn = sqlite3.connect('survey_data.db')
    cursor = conn.cursor()

    # 用 SQL 統計各個星座填寫的人數
    cursor.execute('SELECT zodiac, COUNT(*) FROM results GROUP BY zodiac')
    zodiac_counts = cursor.fetchall()

    # 用 SQL 統計各個 DISC 人格填寫的人數
    cursor.execute('SELECT disc, COUNT(*) FROM results GROUP BY disc')
    disc_counts = cursor.fetchall()

    conn.close()

    return jsonify({
        "zodiac_stats": zodiac_counts,
        "disc_stats": disc_counts
    })

# 啟動 ngrok 與 Flask
ngrok.kill()
public_url = ngrok.connect(5001).public_url
print("==================================================")
print(f"👉 請把這串新網址貼到前端：{public_url}")
print("==================================================")

if __name__ == '__main__':
    app.run(port=5001)

👉 請把這串新網址貼到前端：https://overhead-hatbox-yiddish.ngrok-free.dev
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5001
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [04/Nov/2026 12:38:03] "OPTIONS /submit HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [04/Nov/2026 12:38:03] "POST /submit HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [04/Nov/2026 12:38:10] "OPTIONS /submit HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [04/Nov/2026 12:38:10] "POST /submit HTTP/1.1" 200 -


In [16]:
import sqlite3

# 連線到資料庫
conn = sqlite3.connect('survey_data.db')
cursor = conn.cursor()

print("===== 星座統計結果 =====")
cursor.execute('SELECT zodiac, COUNT(*) FROM results GROUP BY zodiac')
for row in cursor.fetchall():
    print(f"星座: {row[0]} | 人數: {row[1]}")

print("\n===== DISC 統計結果 =====")
cursor.execute('SELECT disc, COUNT(*) FROM results GROUP BY disc')
for row in cursor.fetchall():
    print(f"DISC: {row[0]} | 人數: {row[1]}")

conn.close()

===== 星座統計結果 =====
星座: 天秤座 | 人數: 1
星座: 牡羊座 | 人數: 1
星座: 獅子座 | 人數: 1
星座: 雙魚座 | 人數: 1

===== DISC 統計結果 =====
DISC: C型 (分析型 Conscientiousness) | 人數: 1
DISC: D型 (支配型 Dominance) | 人數: 1
DISC: S型 (穩健型 Steadiness) | 人數: 2
